In [ ]:
# ============================================================
# TopologicPy Provenance.Compose demonstration
# Merge -> Difference -> composed provenance graph
# ============================================================

from collections import Counter

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from topologicpy.Cell import Cell
from topologicpy.Vertex import Vertex
from topologicpy.Topology import Topology
from topologicpy.Provenance import Provenance
from topologicpy.Plotly import Plotly
from topologicpy.TGraph import TGraph


# ------------------------------------------------------------
# 1. Create two overlapping prisms
# ------------------------------------------------------------

p1 = Cell.Prism(
    origin=Vertex.ByCoordinates(0, 0, 0)
)

p2 = Cell.Prism(
    origin=Vertex.ByCoordinates(0.6, 0.6, 0.6)
)


# ------------------------------------------------------------
# 2. First operation: Merge
# ------------------------------------------------------------

merged, provenance_1 = Topology.Merge(
    p1,
    p2,
    returnProvenance=True,
)

assert merged is not None
assert provenance_1.supported


# ------------------------------------------------------------
# 3. Second operation: Difference
#
# Notice that the output of Merge becomes the first operand of
# Difference. This is the boundary that Provenance.Compose
# should stitch together.
# ------------------------------------------------------------

result, provenance_2 = Topology.Difference(
    merged,
    p2,
    returnProvenance=True,
)

assert result is not None
assert provenance_2.supported


# ------------------------------------------------------------
# 4. Compose the two provenance histories
# ------------------------------------------------------------

provenance = Provenance.Compose(
    provenance_1,
    provenance_2,
)

provenance_graph = provenance.Graph(
    topologyType="Face"
)


# ------------------------------------------------------------
# 5. Inspect the result
# ------------------------------------------------------------

vertices = TGraph.Vertices(
    provenance_graph,
    asTopologic=False,
    active=True,
) or []

edges = TGraph.Edges(
    provenance_graph,
    asTopologic=False,
    active=True,
) or []

role_counts = Counter(
    (v.get("dictionary") or {}).get("role")
    for v in vertices
)

print("Composed provenance")
print("-------------------")
print("Stages:          ", provenance.metadata.get("stageCount"))
print("Operations:      ", provenance.metadata.get("operations"))
print("Boundary matches:", provenance.metadata.get("boundaryMatches"))
print("Graph vertices:  ", len(vertices))
print("Graph edges:     ", len(edges))
print("Roles:           ", dict(role_counts))
print("Final Faces:     ", len(Topology.Faces(result, silent=True)))


# ============================================================
# VISUALISATION
# ============================================================

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{"type": "scene"}, {"type": "scene"}],
        [{"type": "scene"}, {"type": "scene"}],
    ],
    subplot_titles=[
        "Inputs",
        "After Merge",
        "After Difference",
        "Composed Face Provenance",
    ],
    horizontal_spacing=0.02,
    vertical_spacing=0.06,
)


# ------------------------------------------------------------
# Helper: add a TopologicPy topology to a subplot
# ------------------------------------------------------------

def add_topology(
    topology,
    row,
    col,
    *,
    face_color,
    opacity=0.45,
    edge_color="black",
):
    data = Plotly.DataByTopology(
        topology,
        showVertices=False,
        showEdges=True,
        edgeColor=edge_color,
        edgeWidth=2,
        showFaces=True,
        faceColor=face_color,
        faceOpacity=opacity,
        silent=True,
    ) or []

    for trace in data:
        trace.showlegend = False
        fig.add_trace(
            trace,
            row=row,
            col=col,
        )


# ------------------------------------------------------------
# 6. Geometry panels
# ------------------------------------------------------------

# Original operands
add_topology(
    p1,
    1,
    1,
    face_color="#4C78A8",
    opacity=0.42,
)

add_topology(
    p2,
    1,
    1,
    face_color="#E45756",
    opacity=0.42,
)

# Merge result
add_topology(
    merged,
    1,
    2,
    face_color="#F2CF5B",
    opacity=0.55,
)

# Difference result
add_topology(
    result,
    2,
    1,
    face_color="#54A24B",
    opacity=0.60,
)


# ============================================================
# 7. Construct a clean staged layout for the provenance graph
#
# x = 0 : source
# x = 1 : intermediate
# x = 2 : final result
# ============================================================

ROLE_X = {
    "source": 0.0,
    "intermediate": 1.0,
    "result": 2.0,
}

ROLE_COLOR = {
    "source": "#4C78A8",
    "intermediate": "#F2CF5B",
    "result": "#54A24B",
}

ROLE_LABEL = {
    "source": "Source",
    "intermediate": "Intermediate",
    "result": "Result",
}


# Group graph vertices by semantic role
by_role = {
    "source": [],
    "intermediate": [],
    "result": [],
}

for record in vertices:
    d = record.get("dictionary") or {}
    role = d.get("role")

    if role in by_role:
        by_role[role].append(record)


# ------------------------------------------------------------
# Stable vertical placement within each provenance stage
# ------------------------------------------------------------

positions = {}

for role in ("source", "intermediate", "result"):

    records = by_role[role]
    n = len(records)

    if n == 0:
        continue

    # Centre each column around y = 0
    spacing = 1.0
    y0 = -0.5 * (n - 1) * spacing

    for i, record in enumerate(records):

        index = record.get("index")

        positions[index] = (
            ROLE_X[role],
            y0 + i * spacing,
            0.0,
        )


# ------------------------------------------------------------
# 8. Provenance edges
# ------------------------------------------------------------

RELATION_COLOR = {
    "unchanged": "#888888",
    "modified": "#E45756",
    "generated": "#72B7B2",
    "deleted": "#B279A2",
}

edge_groups = {}

for edge in edges:

    src = edge.get("src")
    dst = edge.get("dst")

    if src not in positions or dst not in positions:
        continue

    d = edge.get("dictionary") or {}

    relation = str(
        d.get("relation", "unknown")
    ).lower()

    operation = d.get("operation", "")
    stage = d.get("provenanceStage", "")

    edge_groups.setdefault(
        relation,
        {
            "x": [],
            "y": [],
            "z": [],
            "hover": [],
        },
    )

    a = positions[src]
    b = positions[dst]

    group = edge_groups[relation]

    group["x"].extend([
        a[0],
        b[0],
        None,
    ])

    group["y"].extend([
        a[1],
        b[1],
        None,
    ])

    group["z"].extend([
        a[2],
        b[2],
        None,
    ])

    text = (
        f"<b>{relation}</b>"
        f"<br>Operation: {operation}"
        f"<br>Stage: {stage}"
    )

    group["hover"].extend([
        text,
        text,
        None,
    ])


for relation, group in edge_groups.items():

    fig.add_trace(
        go.Scatter3d(
            x=group["x"],
            y=group["y"],
            z=group["z"],
            mode="lines",
            line=dict(
                width=3,
                color=RELATION_COLOR.get(
                    relation,
                    "#666666",
                ),
            ),
            name=relation.capitalize(),
            legendgroup=f"relation-{relation}",
            hovertext=group["hover"],
            hovertemplate="%{hovertext}<extra></extra>",
        ),
        row=2,
        col=2,
    )


# ------------------------------------------------------------
# 9. Provenance vertices
# ------------------------------------------------------------

for role in ("source", "intermediate", "result"):

    records = by_role[role]

    if not records:
        continue

    xs = []
    ys = []
    zs = []
    labels = []
    hover = []

    for record in records:

        index = record.get("index")
        d = record.get("dictionary") or {}

        if index not in positions:
            continue

        x, y, z = positions[index]

        xs.append(x)
        ys.append(y)
        zs.append(z)

        labels.append(str(index))

        provenance_id = d.get(
            "provenanceId",
            "",
        )

        topology_type = d.get(
            "topologyType",
            "",
        )

        source_roles = d.get(
            "sourceRoles",
            [],
        )

        hover.append(
            f"<b>{ROLE_LABEL[role]}</b>"
            f"<br>Graph index: {index}"
            f"<br>Topology: {topology_type}"
            f"<br>Provenance ID: {provenance_id}"
            f"<br>Source roles: {source_roles}"
        )

    fig.add_trace(
        go.Scatter3d(
            x=xs,
            y=ys,
            z=zs,
            mode="markers",
            marker=dict(
                size=8,
                color=ROLE_COLOR[role],
                line=dict(
                    width=1,
                    color="black",
                ),
            ),
            name=ROLE_LABEL[role],
            legendgroup=f"role-{role}",
            hovertext=hover,
            hovertemplate="%{hovertext}<extra></extra>",
        ),
        row=2,
        col=2,
    )


# ------------------------------------------------------------
# 10. Stage labels
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter3d(
        x=[0, 1, 2],
        y=[
            max(
                [p[1] for p in positions.values()],
                default=0,
            ) + 2
        ] * 3,
        z=[0, 0, 0],
        mode="text",
        text=[
            "SOURCES",
            "MERGE OUTPUT",
            "FINAL RESULT",
        ],
        textfont=dict(size=14),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=2,
    col=2,
)


# ------------------------------------------------------------
# 11. Figure formatting
# ------------------------------------------------------------

fig.update_scenes(
    xaxis=dict(
        visible=False,
        showgrid=False,
        zeroline=False,
    ),
    yaxis=dict(
        visible=False,
        showgrid=False,
        zeroline=False,
    ),
    zaxis=dict(
        visible=False,
        showgrid=False,
        zeroline=False,
    ),
    aspectmode="data",
)

# Look directly at the provenance graph plane
fig.update_layout(
    scene4_camera=dict(
        eye=dict(
            x=0,
            y=0,
            z=2.5,
        ),
        projection=dict(
            type="orthographic"
        ),
    ),
    width=1500,
    height=1050,
    title=dict(
        text=(
            "TopologicPy Provenance.Compose — "
            "Merge → Difference"
        ),
        x=0.5,
    ),
    margin=dict(
        l=10,
        r=10,
        t=80,
        b=10,
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.01,
        xanchor="center",
        x=0.5,
    ),
)

fig.show()